### Conventions

- $\theta$ is the namedtuple containing all diffsky bounded parameters, i.e., [`ParamCollection`](https://github.com/ArgonneCPAC/diffsky/blob/1ff1b9e47b35daa574f41d4472730a1d2a7f8b9b/diffsky/param_utils/diffsky_param_wrapper_merging.py#L27).

  I refer to it as `param_coll`.

- $\theta^*$ is the namedtuple containing all diffsky unbounded parameters, i.e., [`UParamCollection`](https://github.com/ArgonneCPAC/diffsky/blob/1ff1b9e47b35daa574f41d4472730a1d2a7f8b9b/diffsky/param_utils/diffsky_param_wrapper_merging.py#L46).

  I refer to it as `uparam_coll`.

- $\bar{\theta}$ is the flat version of `ParamCollection`: [`DiffskyParamsFlat`](https://github.com/ArgonneCPAC/diffsky/blob/1ff1b9e47b35daa574f41d4472730a1d2a7f8b9b/diffsky/param_utils/diffsky_param_wrapper_merging.py#L418). This is computed from $\theta$ using `dpwm.unroll_param_collection_into_flat_array`.

  I refer to it as `param_flat`.

- $\bar{\theta}^*$ is the flat version of `UParamCollection`: [`DiffskyUParamsFlat`](https://github.com/ArgonneCPAC/diffsky/blob/1ff1b9e47b35daa574f41d4472730a1d2a7f8b9b/diffsky/param_utils/diffsky_param_wrapper_merging.py#L420). This is computed from $\theta^*$ using `dpwm.unroll_u_param_collection_into_flat_array`.

  I refer to it as `uparam_flat`.

- The subscript $_{\rm var}$ refers to the subset of parameters that we are inferring, i.e., that we are **varying**. We typically compute gradients of the loss/likelihood/log-density defined as a function of **$\bar{\theta}^*_{\rm var}$**, the flat representation of the subset of varied unbounded parameters.

  I refer to $\bar{\theta}^*_{\rm var}$ as `var_uparam_flat` and $\bar{\theta}_{\rm var}$ as `var_param_flat`.

### Basic steps
- load target data
- load diffsky parameters
- get lc_data_phot
- define a likelihood and create ``loss_data``



In [ ]:
%cd /home/nvilla/diffsky

# Libraries

In [ ]:
import jax
import jax.numpy as jnp
import numpy as np
import matplotlib.pyplot as plt

from diffsky.experimental.lc_generators.lc_phot import mc_lc_phot
from diffsky.param_utils import diffsky_param_wrapper_merging as dpwm
from diffsky.soft_histograms.signdhist_lomem import nnsig_ndhist

from diffsky.experimental.inference import utils, fisher, prior, likelihood

ran_key = jax.random.key(42)

### Set `dir_out`

To store outputs.

In [ ]:
dir_out = '/home/nvilla/diffstuff_experiments/hmc_dev/scripts_hmc/output'

# Get LC data phot

In [ ]:
from dsps.cosmology import flat_wcdm
from dsps.data_loaders import load_ssp_templates, load_transmission_curve
from diffsky.experimental.lc_generators.lc_data_phot import weighted_lc_data_phot

In [ ]:
# -- Settings --

num_halos = 500
z_min = 0.1
z_max = 0.2
lgmp_min = 10.5
lgmp_max = 15.0
sky_area_degsq = 1

ran_key, lc_data_key = jax.random.split(ran_key, 2)

# Cosmology
cosmo_params = flat_wcdm.PLANCK15
fb = 0.156

n_z_phot_table = 15

# Transmission curves data
filter_names = ["u", "g", "r", "i", "z"]
tcurves_args = {
    "fn": None,
    "bn_pat": "sdss_{}_transmission.h5",
    "drn": "/home/nvilla/diffstuff_data/filters",
}

# SSP data arguments
ssp_data_args = {
    "fn": None,
    "drn": "/home/nvilla/diffstuff_data",
    "bn": "ssp_data_fsps_v3.2_lgmet_age.h5",
}



# -- Get lc_data_phot --

# Load SSP data
ssp_data = load_ssp_templates(**ssp_data_args)

# Load transmission curve data (wavelength array and transmission)
bn_list = [tcurves_args["bn_pat"].format(x) for x in filter_names]
tcurves = [
    load_transmission_curve(
        fn=tcurves_args["fn"], bn_pat=bn, drn=tcurves_args["drn"]
    )
    for bn in bn_list
]

# Define a redshift table used for photometry interpolation
z_phot_table = jnp.linspace(z_min, z_max, n_z_phot_table)

# Get weighted LC data
lc_data_phot = weighted_lc_data_phot(
    lc_data_key,
    num_halos,
    z_min,
    z_max,
    lgmp_min,
    lgmp_max,
    sky_area_degsq=sky_area_degsq,
    ssp_data=ssp_data,
    tcurves=tcurves,
    z_phot_table=z_phot_table,
    cosmo_params=cosmo_params,
    logmp_cutoff=11.0,
)

gal_weight = lc_data_phot.halo_weight

# Load diffsky parameters

In [ ]:
# Choose param. collection
param_coll = dpwm.DEFAULT_PARAM_COLLECTION

In [ ]:
# Choose varied parameters
var_uparams_list = [
    "u_mean_ulgm_mseq_ytp",
    "u_mean_ulgy_qseq_ytp"
]

var_params_list = [utils.bounded_name(name) for name in var_uparams_list]

In [ ]:
uparam_coll = dpwm.get_u_param_collection_from_param_collection(*param_coll)
uparam_flat = dpwm.unroll_u_param_collection_into_flat_array(*uparam_coll)

var_uparam_flat = utils.get_var_param_flat_from_param_flat(uparam_flat, var_uparams_list)

# List of indices of varied parameters in the flatten namedtuple
var_flat_idx = utils.compute_varied_params_indices(var_uparam_flat, uparam_flat)

# Load target data

In [ ]:
ran_key, target_data_key = jax.random.split(ran_key, 2)
fake_data, _, _ = mc_lc_phot(target_data_key, lc_data_phot, mc_merge=0, param_collection=param_coll)
target_mags = fake_data.obs_mags_weighted

# Define likelihood

In [ ]:
def target_space_fn(mags):
    return mags[:, 2]


def loglikelihood_from_param_coll(param_coll, loss_data):

    lc_data, XHIST_TARGET, XBINS, loss_key = loss_data
    loss_key, phot_key = jax.random.split(loss_key, 2)

    # Get phot. lightcone
    phot_kern_results, phot_randoms, merging_randoms = mc_lc_phot(
        phot_key,
        lc_data,
        mc_merge=0,
        param_collection=param_coll,
    )
    pred_mags = phot_kern_results.obs_mags_weighted

    # Compute pred. diff. hist
    pred_data = target_space_fn(pred_mags)
    # gal_weight = lc_data.halo_weight
    # gal_weight_masked = get_masked_gal_weight(pred_mags, gal_weight)
    XHIST_PRED = likelihood.soft_xhist(pred_data, XBINS)

    # Compute log-likelihood
    logpdf = likelihood._poisson_kern(XHIST_PRED, XHIST_TARGET)
    
    return logpdf

Compute target soft histogram

In [ ]:
target_data = target_space_fn(target_mags)

NBINS = 30
XBOUNDS = (10.0, 40.0)
XBINS = np.linspace(*XBOUNDS, NBINS)[:-1]

XHIST_TARGET = likelihood.soft_xhist(target_data, XBINS)

Get loss data

In [ ]:
ran_key, loss_key = jax.random.split(ran_key, 2)
loss_data = lc_data_phot, XHIST_TARGET, XBINS, loss_key

Visualize histogram

In [ ]:
XHIST_TARGET, __ = jnp.histogram(target_data, bins=XBINS)
XHIST_TARGET = likelihood.soft_xhist(target_data, XBINS)

fig, ax = plt.subplots(1, 1)
__=ax.plot(XBINS[1:], XHIST_TARGET,
           label='standard histogram')
__=ax.plot(XBINS[1:], XHIST_TARGET,'--',
           label='soft histogram')
leg = ax.legend()

# Define flat partial functions

Gradients will be computed with respect to these functions

In [ ]:
def flat_loglikelihood_fn(var_uparam_flat, diffsky_params, loss_data):

    # \theta* from \theta-*_var
    uparam_coll = utils.get_uparam_coll_from_var_uparam_flat(
        var_uparam_flat, diffsky_params
    )
    # \theta from \theta*
    param_coll = dpwm.get_param_collection_from_u_param_collection(*uparam_coll)

    return loglikelihood_from_param_coll(param_coll, loss_data)

In [ ]:
from functools import partial

flat_logprior = partial(
    prior.flat_logprior_fn,
    diffsky_params=uparam_flat,
    var_flat_idx=var_flat_idx,
)

# Compute Fisher Matrix

In [ ]:
eval_point = var_uparam_flat

res = fisher.compute_fisher_matrix(
    eval_point,
    flat_loglikelihood_fn,
    flat_logprior,
    include_prior=False,
    verbose=True,
    diffsky_params=uparam_flat,
    loss_data=loss_data,
)

In [ ]:
np.save(f"{dir_out}/covariance_matrix.npy", res.covariance_matrix)

# Sample from Laplace Approximation

In [ ]:
ran_key, fisher_key = jax.random.split(ran_key, 2)
num_samples = 1000

cov = res.covariance_matrix

samples = jax.random.multivariate_normal(
    fisher_key, mean=jnp.asarray(eval_point), cov=cov, shape=int(num_samples)
)

In [ ]:
samples

# Save samples

In [ ]:
# build dictionary with flat samples from the Laplace approximation
samples_dict = dict(zip(var_uparams_list, samples.T))

# build flat namedtuple with samples
uparam_flat_samples = uparam_flat._replace(**samples_dict)
# get uniform depth pytree
uparam_flat_samples = utils.get_flat_params_all_same_shape(uparam_flat_samples)
# convert into collection
uparam_coll_samples = dpwm.get_u_param_collection_from_u_param_array(uparam_flat_samples)
# convert into bounded
param_coll_samples = dpwm.get_param_collection_from_u_param_collection(*uparam_coll_samples)

# Save ParamCollection with samples
# from diffsky.data_loaders.hacc_utils.lc_mock import write_diffsky_param_collection_merging
# write_diffsky_param_collection_merging(dir_out, "fisher_samples", param_coll)

# Visualize samples

In [ ]:
# from diffsky.data_loaders.hacc_utils.lc_mock import load_diffsky_param_collection_merging
# param_coll_samples = load_diffsky_param_collection_merging(dir_out, "fisher_samples")

param_flat_samples = dpwm.unroll_param_collection_into_flat_array(*param_coll_samples)

In [ ]:
default_flat = dpwm.unroll_param_collection_into_flat_array(*dpwm.DEFAULT_PARAM_COLLECTION)

plt.scatter(param_flat_samples.mean_ulgm_mseq_ytp, param_flat_samples.mean_ulgy_qseq_ytp, s=1)
plt.axvline(default_flat.mean_ulgm_mseq_ytp, color='k', ls='--')
plt.axhline(default_flat.mean_ulgy_qseq_ytp, color='k', ls='--')
plt.show()

# ppd

In [ ]:
def get_photometry(ran_key, param_coll, lc_data):
    
    phot_kern_results, phot_randoms, merging_randoms = mc_lc_phot(
        ran_key,
        lc_data,
        mc_merge=0,
        param_collection=param_coll,
    )

    return phot_kern_results.obs_mags_weighted

In [ ]:
individual_param_coll = utils.unpack_nested_samples(param_coll_samples)

N = 100
ran_key, block_key = jax.random.split(ran_key, 2)
subkeys = jax.random.split(block_key, N)

pred_mags_list = []
diffstarpop_params_list = []
for sample_ind in range(N):
    single_param_coll = individual_param_coll[sample_ind]
    pred_mags = get_photometry(subkeys[sample_ind], single_param_coll, lc_data_phot)

    pred_mags_list.append(pred_mags)
    diffstarpop_params_list.append(single_param_coll.diffstarpop_params)

In [ ]:
plt.figure(figsize=(4, 4))

for m in pred_mags_list:
    m_hist, _ = np.histogram(m, bins=XBINS)
    plt.plot(m_hist, color='gray', alpha=0.25)

plt.show()

In [ ]:
from diffsky.experimental.diagnostics.check_smhm import plot_diffstarpop_insitu_smhm_multi
plot_diffstarpop_insitu_smhm_multi(diffstarpop_params_list)

Compare with prior

In [ ]:
from diffsky.experimental.diagnostics.check_smhm import plot_diffstarpop_insitu_smhm_multi

In [ ]:
n_samples = 100
ref_params = dpwm.DEFAULT_PARAM_COLLECTION

ran_key, prior_key = jax.random.split(ran_key, 2)
individual_samples_prior = prior.sample_from_hard_prior(prior_key, n_samples, ref_params, var_params_list)

pred_mags_list = []
diffstarpop_params_list = []
for sample_ind in range(100):
    single_param_coll = individual_samples_prior[sample_ind]
    
    diffstarpop_params_list.append(single_param_coll.diffstarpop_params)

In [ ]:
plot_diffstarpop_insitu_smhm_multi(diffstarpop_params_list)